# 🎙️ All-in-One AI Audio Studio (종합 오디오 생성 플랫폼)
### Connect AI Lab × Resemble AI Chatterbox × Qwen3-TTS

내 목소리 10초~30초 녹음으로 **한국어 정밀 나레이션(Qwen3)** 및 **23개 언어 글로벌 더빙(Chatterbox)**을 Colab 무료 GPU에서 일괄 제작합니다.

---
**[통합 파이프라인 안내]**
- **PART 1**: Chatterbox 23개 언어 원어민 억양 글로벌 더빙 (영어, 스페인어, 프랑스어, 일본어, 중국어 등)
- **PART 2**: Qwen3-TTS 고정밀 한국어 팟캐스트 / 장문 스토리텔링 나레이션
- **PART 3**: 원클릭 다국어 일괄 합성 & ZIP 일괄 다운로드

## 📦 [Step 1] 통합 환경 설정 및 라이브러리 설치
Chatterbox TTS 및 Qwen3-TTS, 오디오 처리 엔진(soundfile, ffmpeg)을 설치합니다.

In [ ]:
# ① 통합 엔진 설치
!pip install -q chatterbox-tts "setuptools<81" qwen-tts soundfile

import os, torch
print("CUDA GPU 활성화 상태:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("사용 GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ 상단 메뉴 [런타임] -> [런타임 유형 변경]에서 GPU를 선택해 주세요!")

FLAG = "/content/.audio_setup_done"
if not os.path.exists(FLAG):
    open(FLAG, "w").close()
    print("\n[1단계 완료] 세션을 자동 재시작합니다. 잠시 후 2단계부터 순서대로 실행하세요.")
    os.kill(os.getpid(), 9)
else:
    print("\n환경 준비 완료! 2단계로 진행하세요.")


## 🎙️ [Step 2] 내 목소리 샘플 업로드 (10초~30초)
조용한 방에서 녹음된 내 목소리(mp3, wav, m4a 등)를 업로드합니다.

In [ ]:
import os, subprocess
from google.colab import files
from IPython.display import Audio, display

OUT_DIR = "/content/audio_output"
os.makedirs(OUT_DIR, exist_ok=True)

print("당신의 목소리 파일(10초~30초)을 업로드하세요 (mp3/wav/m4a):")
uploaded = files.upload()
raw_voice_file = list(uploaded.keys())[0]

MY_VOICE = "/content/my_voice_clean.wav"
# 24kHz 모노 오디오 표준 규격으로 자동 변환
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", raw_voice_file, "-ac", "1", "-ar", "24000", "-t", "30", MY_VOICE], check=True)

print("\n✅ 내 목소리 전처리 완료!")
display(Audio(MY_VOICE))


## 🌍 [Pipeline A] Chatterbox 다국어 글로벌 더빙 (23개 언어)
음색은 내 목소리를 유지하되, 각 언어 원어민의 자연스러운 억양으로 발화합니다.

In [ ]:
# Chatterbox 다국어 엔진 로드
import torchaudio
from chatterbox.mtl_tts import ChatterboxMultilingualTTS, SUPPORTED_LANGUAGES

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cb_model = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)
print(f"Chatterbox 로드 완료 ({DEVICE}). 지원 언어:", ", ".join(sorted(SUPPORTED_LANGUAGES)))

# 단일 언어 테스트 (원하는 언어 코드: en, ja, es, fr, zh, de, hi 등)
TARGET_LANG = "en"
SPEECH_TEXT = "Hello! This is my own voice, speaking natural English with Resemble AI Chatterbox."

wav = cb_model.generate(
    SPEECH_TEXT,
    language_id=TARGET_LANG,
    audio_prompt_path=MY_VOICE,
    exaggeration=0.5, # 감정 강도 (0.3~0.8)
    cfg_weight=0.5    # 속도/차분함 제어 (낮출수록 차분)
)
out_path = f"{OUT_DIR}/chatterbox_{TARGET_LANG}.wav"
torchaudio.save(out_path, wav.cpu(), cb_model.sr)
print(f"\n[Chatterbox {TARGET_LANG}] 생성 완료:")
display(Audio(out_path))


## 🇰🇷 [Pipeline B] Qwen3-TTS 한국어 고정밀 장문 팟캐스트 나레이션
내 한국어 억양과 호흡을 정밀 복제하여 장문 다큐멘터리/팟캐스트 대본을 읽습니다.

In [ ]:
import soundfile as sf
from qwen_tts import Qwen3TTSModel

dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
qwen_model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-0.6B-Base", device_map="cuda:0" if torch.cuda.is_available() else "cpu", dtype=dtype)
print("Qwen3-TTS 로드 완료!")

# 내가 녹음할 때 실제로 말했던 문장
REF_SCRIPT = "안녕하세요. 인공지능 오디오 스튜디오 테스트 녹음입니다. 제 목소리로 다양한 콘텐츠를 만듭니다."

# 생성하고자 하는 긴 한국어 나레이션 / 팟캐스트 대본
STORY_SCRIPT = """우리는 늘 무언가를 쫓듯, 쉴 틈 없이 바쁘게만 살아가는 것 같아요.
하지만 지금부터 딱 1분만, 온전히 나 자신을 위한 시간을 내어보세요.
가끔은 이렇게 멈춰 서는 것만으로도, 다시 나아갈 충분한 힘이 생기니까요.
오늘 하루도 정말 수고 많으셨습니다."""

wavs, sr = qwen_model.generate_voice_clone(
    text=STORY_SCRIPT,
    language="Korean",
    ref_audio=MY_VOICE,
    ref_text=REF_SCRIPT
)
qwen_out = f"{OUT_DIR}/qwen_korean_narration.wav"
sf.write(qwen_out, wavs[0], sr)
print("\n[Qwen3 한국어 나레이션] 생성 완료:")
display(Audio(qwen_out))


## 🌐 [Pipeline C] 글로벌 5개국어 일괄 합성 & ZIP 다운로드
영어, 스페인어, 프랑스어, 중국어, 힌디어로 내 목소리를 한 번에 더빙하고 일괄 내려받습니다.

In [ ]:
import shutil

MULTILINGUAL_SCRIPTS = {
    "en": "Hi, this is my own voice, now speaking natural English with global AI dubbing.",
    "es": "Hola, esta es mi propia voz, ahora hablando español con doblaje de IA.",
    "fr": "Bonjour, ceci est ma propre voix, qui parle maintenant français grâce au doublage IA.",
    "zh": "大家好，这是我自己的声音，正在用 AI 进行中文配音。",
    "hi": "नमस्ते, यह मेरी अपनी आवाज़ है, जो अब एआई डबिंग के साथ स्वाभाविक रूप से बोल रही है।"
}

for lang_code, text in MULTILINGUAL_SCRIPTS.items():
    print(f"[{lang_code}] 생성 중: {text}")
    m_wav = cb_model.generate(text, language_id=lang_code, audio_prompt_path=MY_VOICE)
    f_path = f"{OUT_DIR}/{lang_code}_dubbed.wav"
    torchaudio.save(f_path, m_wav.cpu(), cb_model.sr)
    display(Audio(f_path))

# 전체 결과 ZIP 압축 및 자동 다운로드
zip_filename = "/content/my_global_voice_pack"
shutil.make_archive(zip_filename, "zip", OUT_DIR)
files.download(f"{zip_filename}.zip")
print("\n🎉 모든 글로벌 오디오 생성이 완료되어 ZIP으로 다운로드되었습니다!")
